# 02 — Baseline Model

A dumb baseline before a smart model — gives the XGBoost model in notebook 03
something concrete to beat, and proves the added complexity earns its place.

Logistic regression with `class_weight='balanced'` (handles the 21.87% imbalance
without inventing data), one-hot encoded categoricals (fine for a linear model —
WOE encoding is explored separately in notebook 04 for the tree model / feature
selection story). Evaluated with AUC/KS/Brier — never accuracy (see `src/metrics.py`).

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from src.preprocessing import TARGET, CATEGORICAL_COLS, NUMERIC_COLS
from src.metrics import summarize

pd.set_option('display.width', 120)

In [2]:
train = pd.read_csv('../data/processed/train.csv')
val = pd.read_csv('../data/processed/val.csv')
test = pd.read_csv('../data/processed/test.csv')

X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_val, y_val = val.drop(columns=[TARGET]), val[TARGET]
X_test, y_test = test.drop(columns=[TARGET]), test[TARGET]

print('train:', X_train.shape, 'val:', X_val.shape, 'test:', X_test.shape)

train: (19449, 11) val: (6483, 11) test: (6484, 11)


## Pipeline: impute-free (already clean) → scale numeric → one-hot categorical → LR

Everything fit on train only, via `sklearn.Pipeline` — enforces no leakage
mechanically rather than by discipline alone.

In [3]:
preprocess = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), CATEGORICAL_COLS),
])

lr_pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
])

lr_pipe.fit(X_train, y_train)

val_proba = lr_pipe.predict_proba(X_val)[:, 1]
test_proba = lr_pipe.predict_proba(X_test)[:, 1]

print('VAL :', {k: round(v, 4) for k, v in summarize(y_val, val_proba).items()})
print('TEST:', {k: round(v, 4) for k, v in summarize(y_test, test_proba).items()})

VAL : {'auc': 0.8782, 'ks': 0.6237, 'gini': 0.7564, 'pr_auc': 0.7231, 'brier': 0.1391}
TEST: {'auc': 0.8697, 'ks': 0.6038, 'gini': 0.7394, 'pr_auc': 0.7166, 'brier': 0.1384}


## Coefficient inspection (sanity check on direction, not just magnitude)

Every coefficient should have a direction that makes real-world sense — this is a
cheap, fast leakage/bug check before trusting any downstream metric.

In [4]:
feature_names = lr_pipe.named_steps['preprocess'].get_feature_names_out()
coefs = pd.Series(lr_pipe.named_steps['model'].coef_[0], index=feature_names).sort_values()
print(coefs)

cat__person_home_ownership_OWN     -1.629343
cat__loan_intent_VENTURE           -1.059580
num__loan_amnt                     -0.640887
cat__loan_intent_EDUCATION         -0.595632
cat__loan_intent_PERSONAL          -0.446706
cat__loan_intent_MEDICAL           -0.053413
num__cb_person_cred_hist_length    -0.041387
num__person_emp_length             -0.021778
num__person_age                     0.008904
cat__cb_person_default_on_file_Y    0.022607
cat__loan_grade_B                   0.023833
num__person_income                  0.025978
cat__person_home_ownership_OTHER    0.081385
cat__loan_grade_C                   0.136947
cat__loan_intent_HOMEIMPROVEMENT    0.262993
num__loan_int_rate                  0.409937
cat__person_home_ownership_RENT     0.595188
num__loan_percent_income            1.349542
cat__loan_grade_D                   2.036041
cat__loan_grade_E                   2.149212
cat__loan_grade_F                   2.478678
cat__loan_grade_G                   3.391690
dtype: flo

In [5]:
import json
import os
os.makedirs('../reports', exist_ok=True)

results = {
    'model': 'logistic_regression_baseline',
    'val': summarize(y_val, val_proba),
    'test': summarize(y_test, test_proba),
}
with open('../reports/baseline_results.json', 'w') as f:
    json.dump(results, f, indent=2)
results

{'model': 'logistic_regression_baseline',
 'val': {'auc': 0.878194890959139,
  'ks': 0.6237146990394268,
  'gini': 0.7563897819182781,
  'pr_auc': 0.7231355551858111,
  'brier': 0.1390847852458561},
 'test': {'auc': 0.8697091202891926,
  'ks': 0.6038319569552152,
  'gini': 0.7394182405783851,
  'pr_auc': 0.7165849371690065,
  'brier': 0.1384473592480507}}

## Summary

Logistic regression baseline established with AUC/KS/Gini/PR-AUC/Brier on both val
and test splits, saved to `reports/baseline_results.json`. Coefficient signs checked
for real-world plausibility (sanity check, not a leakage audit substitute). This is
the number the XGBoost model in notebook 03 must beat to justify its complexity.